# DREAM AI — Kaggle GPU boot
Enable **GPU** and **Internet** in notebook settings, then choose **Run All**. No dataset input is needed.
This downloads Qwen3-1.7B weights and tests local generation. It does not yet fine-tune the model or create an always-on server.


In [ ]:
!pip -q install -U transformers accelerate safetensors


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
assert torch.cuda.is_available(), 'Turn on GPU in Kaggle Settings first.'
model_id = 'Qwen/Qwen3-1.7B'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map='cuda:0')
messages = [{'role':'user','content':'You are DREAM AI. Introduce yourself and explain what you can help with.'}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
output = model.generate(**inputs, max_new_tokens=180, do_sample=True, temperature=0.7)
answer = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(answer)
from pathlib import Path
Path('/kaggle/working/dream-ai-first-answer.txt').write_text(answer)
print('Saved: /kaggle/working/dream-ai-first-answer.txt')
